In [1]:
# %run take_split.py

In [2]:
!rm -rf cache/scoring/

In [ ]:
import wandb


wandb.login()


api = wandb.Api()


entity = "loriss"
project = "linear_coder"


runs = api.runs(f"{entity}/{project}")


for run in runs:
    print(f"Deleting run: {run.name} ({run.id})")
    run.delete()


wandb: Currently logged in as: loriss to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Deleting run: MSECoderProjUSimpSparse_LESSEstimator: normalize=True (9a1qzk21)
Deleting run: MSECoderProjUSimpSparse_LESSEstimator: normalize=True (met8nghe)
Deleting run: MSECoderProjUSimpSparse_LESSEstimator: normalize=True (s8z9e393)
Deleting run: MSECoderProjUSimpSparse_LESSEstimator: normalize=True (yrzwjdmn)
Deleting run: MSECoderProjUSimpSparse_LESSEstimator: normalize=True (iuaf9kcl)
Deleting run: CosineCoder_LESSEstimator: normalize=True (dmpl52p0)
Deleting run: CosineCoder_LESSEstimator: normalize=True (yddl6x7w)
Deleting run: MSECoderLemon_LESSEstimator: normalize=True (jg0duvwn)
Deleting run: MSECoderLemon_LESSEstimator: normalize=True (5dum4wpt)


In [ ]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import torch
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)


In [ ]:

import torch
from scipy import stats
import numpy as np

import numpy as np
from scipy import stats
import itertools
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    linear_coders
)
train_dataset, test_dataset, estimators = load_data_and_estimators()


[INFO] PyTorch version 2.7.0+cu126 available.
Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.


influence_estimate_path: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True
influence_estimate_path: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True


In [ ]:
os.path.basename(train_dataset_name)

'tulu-v2-sft-mixture'

In [ ]:
for estimator in estimators:
    print(estimator.get_config_string(), estimator.get_gradient(train_dataset, train_dataset_name, train_dataset_split, 0).shape)

LESSEstimator: normalize=True torch.Size([122880])
DataInfEstimator: fast_implementation=True torch.Size([122880])


In [ ]:
import logging
logging.getLogger().setLevel(logging.WARNING)
from itertools import cycle


In [ ]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback



logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)


n_test = 1
test_indices = test_dataset.shuffle(seed=42).select(range(n_test))["indices"]

num_devices = torch.cuda.device_count()




In [ ]:
import evaluation_worker
device_ids = itertools.cycle(range(num_devices))
results = []
import logging


logging.getLogger("wandb").setLevel(logging.ERROR)
import os
os.environ["WANDB_SILENT"] = "true"

for estimator in estimators:
    
    print("getting")

    explanations = [
        explanation_type(row.name, estimator)
        for explanation_type in explanation_types
        for _, row in estimator.influence_estimate.iloc[test_indices].iterrows()
    ]
    print("got explanations", estimator)
        
    partial_results_dir =  os.path.join("./cache/scoring/partial/",
        estimator.get_config_string(),
        os.path.basename(estimator.model_path),
        train_dataset_name,
        train_dataset_split,
        test_dataset_name,
        test_dataset_split,
        "partial")
    with ProcessPoolExecutor(max_workers=8*num_devices) as executor:
        futures = {
            executor.submit(
                evaluation_worker.process_explanation,
                partial_results_dir,
                estimator, explanation, 
                train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, 
                linear_coders,
                next(device_ids),
                ii
            ): explanation for ii, explanation in enumerate(explanations)
        }

        with tqdm(total=len(futures), desc="Explanations", position=0) as pbar:
            for future in as_completed(futures):
                try:
                    # You can add future.result() here if you want to catch exceptions
                    future.result()  
                except Exception as e:
                    logging.error(f"A future failed: {e}\n{traceback.format_exc()}")
                    raise
                finally:
                    pbar.update(1)

getting
got explanations <influence_estimation.less_inf.LESSEstimator object at 0x7ff97de83ee0>


Explanations:   0%|          | 0/5 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/srv/home/users/loriss21cs/cfe/evaluation_worker.py", line 43, in process_explanation
    o.fit()
  File "/srv/home/users/loriss21cs/cfe/linear_coders.py", line 424, in fit
    t_opt = GSHP_tensor(t_unconstrained, t_unconstrained, 1.0, max(1,int((1.0-self.reg_lambda)*self.t.shape)))
TypeError: can't multiply sequence by non-int of type 'float'
Traceback (most recent call last):
  File "/srv/home/users/loriss21cs/cfe/evaluation_worker.py", line 43, in process_explanation
    o.fit()
  File "/srv/home/users/loriss21cs/cfe/linear_coders.py", line 424, in fit
    t_opt = GSHP_tensor(t_unconstrained, t_unconstrained, 1.0, max(1,int((1.0-self.reg_lambda)*self.t.shape)))
TypeError: can't multiply sequence by non-int of type 'float'
Traceback (most recent call last):
  File "/srv/home/users/loriss21cs/cfe/evaluation_worker.py", line 43, in process_explanation
    o.fit()
  File "/srv/home/users/loriss

Process SpawnProcess-2:
Process SpawnProcess-5:
Explanations:  60%|██████    | 3/5 [00:53<00:35, 17.82s/it]Process SpawnProcess-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/usr/lib/python3.10/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
KeyboardInterrupt
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  

Early stopping at step 7400, best_score=-76.214665
Early stopping at step 1100, best_score=0.487975
Early stopping at step 8200, best_score=-76.228194
Early stopping at step 2900, best_score=-82.586601
Early stopping at step 1100, best_score=0.477572
Early stopping at step 2400, best_score=-82.580041
Early stopping at step 1100, best_score=-78.469963
Early stopping at step 1100, best_score=0.447253
Early stopping at step 2900, best_score=-78.436903
Early stopping at step 16000, best_score=-0.006983
Early stopping at step 1100, best_score=0.547384
Early stopping at step 16000, best_score=-0.006983
Early stopping at step 1100, best_score=0.547384


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df = pd.read_parquet("./cache/scoring/partial")
# assert (df.groupby(["explanation_type", "linear_coder"]).count() == n_test).all().all()


In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('pred_gain', 'mean')].sort_values()

In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('l1', 'mean')].sort_values()

In [ ]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('mse', 'mean')].sort_values()

In [ ]:
df

In [ ]:
df[~df["linear_coder"].str.contains("KLT")].groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('pred_gain', 'mean')].sort_values(ascending=False)